In [1]:
import torch
import torch.nn as nn
from models.Unet import DenoiserNetwork_Unet
from models.Data_Load import DataLoader_Diffusion,Dataset_Diffusion
from check import setup_torch
import numpy as np


device = setup_torch()


❌ Using CPU globally


In [ ]:
data = torch.load("../../json_data/dataset_updated.pt", map_location=device)
print(data[0][0].shape)
dataset = Dataset_Diffusion(data,normalize=True)

train_size = 1200
test_size = len(dataset) - train_size
gen = torch.Generator(device=device)
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size], generator=gen
)

dataLoader = DataLoader_Diffusion(train_dataset, batch_size=30, shuffle=True)
dataLoader_test = DataLoader_Diffusion(test_dataset, batch_size=30, shuffle=False)


FileNotFoundError: [Errno 2] No such file or directory: 'dataset_updated.pt'

In [3]:
def train(model, dataLoader, device):
    from models.Unet import converter_class_idx
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    num_epochs = 30

    model.to(device)  
    for epoch in range(num_epochs):
        model.train()
        for batch in dataLoader:
            images, labels = batch
            images = images.to(device)

            labels = [converter_class_idx(label, True) for label in labels]
            labels = torch.tensor(labels, dtype=torch.long, device=device)

            timesteps = torch.randint(0, 1000, (images.size(0),), device=device).long()

            noisy_images = images + 0.1 * torch.randn_like(images)

            optimizer.zero_grad()
            outputs = model(noisy_images, timesteps, labels)
            loss = nn.MSELoss()(outputs, images)
            loss.backward()
            optimizer.step()

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

In [4]:
model = DenoiserNetwork_Unet().to(device)

train(model, dataLoader, device)

/home/charbel/vip/MaqamDiffusion/diffusion/lib/python3.13/site-packages/torch/utils/_device.py:109: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)


Epoch [1/30], Loss: 0.0032
Epoch [2/30], Loss: 0.0033
Epoch [3/30], Loss: 0.0019
Epoch [4/30], Loss: 0.0025
Epoch [5/30], Loss: 0.0033
Epoch [6/30], Loss: 0.0016
Epoch [7/30], Loss: 0.0017
Epoch [8/30], Loss: 0.0021
Epoch [9/30], Loss: 0.0023
Epoch [10/30], Loss: 0.0022
Epoch [11/30], Loss: 0.0018
Epoch [12/30], Loss: 0.0023
Epoch [13/30], Loss: 0.0023
Epoch [14/30], Loss: 0.0016
Epoch [15/30], Loss: 0.0017
Epoch [16/30], Loss: 0.0017
Epoch [17/30], Loss: 0.0016
Epoch [18/30], Loss: 0.0020
Epoch [19/30], Loss: 0.0032
Epoch [20/30], Loss: 0.0019
Epoch [21/30], Loss: 0.0016
Epoch [22/30], Loss: 0.0021
Epoch [23/30], Loss: 0.0026
Epoch [24/30], Loss: 0.0015
Epoch [25/30], Loss: 0.0026
Epoch [26/30], Loss: 0.0017
Epoch [27/30], Loss: 0.0016
Epoch [28/30], Loss: 0.0018
Epoch [29/30], Loss: 0.0022
Epoch [30/30], Loss: 0.0014


In [5]:
from models.Unet import converter_class_idx

model.eval()
with torch.no_grad():
    for batch in dataLoader_test:
        images, labels = batch
        images = images.to(device)

        labels = [converter_class_idx(label, True) for label in labels]
        labels = torch.tensor(labels, dtype=torch.long, device=device)

        timesteps = torch.randint(0, 1000, (images.size(0),), device=device).long()

        noisy_images = images + 0.1 * torch.randn_like(images)

        outputs = model(noisy_images, timesteps, labels)
        loss = nn.MSELoss()(outputs, images)

In [76]:
from matplotlib import pyplot as plt
import matplotlib as mpl
import torch
from models.Unet import converter_class_idx




def test_if_they_match(tested_one):

    current_image = dataset.data[tested_one]
    current_image = current_image.unsqueeze(0).unsqueeze(0).to(device)
    classes = [0, 1, 2, 3, 4, 5, 6, 7]

    model.eval()
    with torch.no_grad():
        timestep = torch.tensor([1000], dtype=torch.long, device=device)
        scores = []
        for y in classes:
            class_label = torch.tensor([y], dtype=torch.long, device=device)
            x0_hat = model(current_image, timestep, class_label)
            score = torch.dot(current_image.flatten(), x0_hat.flatten()) / (torch.norm(current_image.flatten()) * torch.norm(x0_hat.flatten()) + 1e-8)
            scores.append(score.item())

    predicted_class = classes[np.argmin(scores)]
    data_reconstructed_cpu = model(
        current_image, 
        timestep, 
        torch.tensor([predicted_class], device=device)
    ).squeeze(0).squeeze(0).detach().cpu().numpy()



    if predicted_class == converter_class_idx(dataset.labels[tested_one], True):
        return 1
    else :
        return 0

count = 0


music_start_time_list = [0, 8, 18, 25, 100, 107, 129, 136, 150, 202, 219, 241, 254, 262, 271, 279, 286, 307, 320, 334, 353, 368, 390, 424, 438, 447, 457, 466, 475, 491, 505, 520, 534, 544, 552, 568, 589, 598, 619, 648, 666, 683, 690, 699, 707, 717, 725, 741, 749, 757,
                         # ... (rest of the list would be here)
                         ]

for i in music_start_time_list:
    count += test_if_they_match(i)


print("accuracy:", count / len(music_start_time_list))




accuracy: 0.24


104

In [ ]:
import torch

def converter_class_idx(label, to_index=False):
    classes = ['Bayati', 'Hijaz', 'Kurd', 'Nawa', 'Rast', 'Saba', 'Sika', 'Ushaq']
    if to_index:
        return classes.index(label)
    else:
        return classes[label]
    
random_label_vec = 

torch.Size([3, 14883])